# RetinaGuard — Production Baseline Training (Google Colab)

This notebook trains the **RetinaGuard Dual-Head EfficientNet** model on real fundus photographs using the strict group-aware splits (`splits_v2.csv`).

### Pipeline Stages:
1. **GPU Verification**: Ensure an active NVIDIA accelerator is assigned.
2. **Storage Mount & Project Setup**: Mount Google Drive and stage the project at `/content/rg` on local NVMe.
3. **Pre-processing Cache**: Crop black borders and resize to 384x384 directly onto local fast disk.
4. **Training (`train.py`)**: Train on `train` split, validate on `val` split, checkpoint `best.pt` directly into Google Drive.
5. **Validation Evaluation (`evaluate_model.py`)**: Compute QWK, referable sensitivity/specificity, bootstrap CIs, confusion matrix, and ONNX export.
6. **Test Evaluation**: Final independent evaluation cell (run once per release).

In [ ]:
# Cell 1: Hardware Check
import sys
import torch

if not torch.cuda.is_available():
    raise SystemError("No GPU detected! Please change runtime type to GPU (T4 / A100 / L4) under Runtime -> Change runtime type.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"[+] GPU Active: {gpu_name} ({vram_gb:.2f} GB VRAM)")
print(f"[+] PyTorch Version: {torch.__version__} (CUDA: {torch.version.cuda})")

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Ensure Drive output directory exists
DRIVE_RUN_DIR = Path("/content/drive/MyDrive/RetinaGuard/runs/baseline_b0_384")
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"[+] Google Drive checkpoint directory ready: {DRIVE_RUN_DIR}")

In [ ]:
# Cell 3: Stage Project to Local Ephemeral Disk & Install Dependencies
# Stage code to local fast drive /content/rg
!mkdir -p /content/rg
!cp -r /content/drive/MyDrive/RetinaGuard/code/* /content/rg/ 2>/dev/null || cp -r /content/RetinaGuard/* /content/rg/ 2>/dev/null || echo "Staged from local directory"

%cd /content/rg

# Install project dependencies without overwriting Colab's pre-compiled PyTorch CUDA build
!pip install --quiet timm onnx onnxruntime scikit-learn pyyaml imagehash opencv-python-headless

import sys
sys.path.insert(0, "ml/src")
sys.path.insert(0, "backend/src")
print("[+] Environment and dependencies configured.")

In [ ]:
# Cell 4: Build Pre-processed Fundus Cache on Local Ephemeral Disk (384x384)
# Pre-crops black borders and resizes to 384x384 for maximum training throughput
!python ml/scripts/build_cache.py \
    --raw-dir data/raw/aptos/train_images \
    --out-dir /content/cache_384/aptos \
    --size 384 \
    --tol 15

In [ ]:
# Cell 5: Train EfficientNet-B0 Dual-Head Model
# Trains on train split, validates on val split each epoch, NEVER touches test split.
# Checkpoints best.pt directly into Google Drive.
!python ml/scripts/train.py \
    --config configs/baseline_b0_384.yaml \
    --cache-dir /content/cache_384/aptos \
    --splits data/processed/splits_v2.csv \
    --out-dir /content/drive/MyDrive/RetinaGuard/runs/baseline_b0_384

In [ ]:
# Cell 6: Validation Set Evaluation & Artifact Generation
!python ml/scripts/evaluate_model.py \
    --checkpoint /content/drive/MyDrive/RetinaGuard/runs/baseline_b0_384/best.pt \
    --splits data/processed/splits_v2.csv \
    --cache-dir /content/cache_384/aptos \
    --split val \
    --out-dir /content/drive/MyDrive/RetinaGuard/runs/baseline_b0_384/eval_val

---
## Final Unseen Test Split Evaluation (Run Strictly Once Per Model Version)
> [!WARNING]
> The cell below evaluates the model against the held-out **Test split** (`splits_v2.csv`).
> **Do not iterate hyperparameters on this split.** It is intended exclusively for final benchmark reporting.

In [ ]:
# Cell 7: Isolated Test Split Benchmark
!python ml/scripts/evaluate_model.py \
    --checkpoint /content/drive/MyDrive/RetinaGuard/runs/baseline_b0_384/best.pt \
    --splits data/processed/splits_v2.csv \
    --cache-dir /content/cache_384/aptos \
    --split test \
    --out-dir /content/drive/MyDrive/RetinaGuard/runs/baseline_b0_384/eval_test